# EXP-004 - H3: UID Entity Aggregation (the "magic feature")

**Hypothesis H3** (pre-registered): a UID key that identifies the card/account behind each
transaction, plus per-UID aggregates, is the single largest feature block - private LB
delta-AUC >= +0.015 over EXP-003 AND private >= 0.93. This block decided the 2019 competition.

**Feature diff vs EXP-003** (the ONLY change; model and params identical):
- UID key: `card1 _ addr1 _ round(TransactionDT/86400 - D1)` - frequency-encoded
- per-UID aggregates over the train+test UNION (label-free, transductive):
  `uid_count`, `uid_amt_mean`, `uid_amt_std`, `uid_amt_ratio`

**Leakage argument**: the UID key is row-local (all components from the same transaction;
`DT/86400 - D1` cancels absolute time). The aggregates use NO label - purely distributional -
so computing them over the train+test union is transductive, not target leakage. The private
LB (a disjoint later time slice) is an independent leak detector: a label leak would inflate
public and collapse private. Canonical implementations + formal argument:
`src/features/engineering.py`, `exercises/ex03_leakage_and_validation.md`.

**Outputs**: `holdout_pred_exp004.csv` and `submission.csv`. DeLong computed off-notebook.

**Anchors**: EXP-003 LB 0.9284 / 0.8998 (SUB-004).

In [ ]:
import os
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SPLIT_QUANTILE = 0.8
SECONDS_PER_MONTH = 86400 * 30.44
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}
MISSING_TOKEN = "__missing__"
FREQ_NUMERIC_CATS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
D_NORM_COLS = [f"D{i}" for i in range(1, 16) if i != 9]

LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=192,
    min_data_in_leaf=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    seed=42,
    n_jobs=-1,
    verbosity=-1,
)
MAX_ROUNDS = 5000
ES_PATIENCE = 200

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not attached (Add Input -> Competitions).")
    DATA_DIR = hits[0].parent
else:
    DATA_DIR = Path("../../../data/raw")
print(f"Data dir: {DATA_DIR}")

## 0. Feature functions - inline mirrors of `src/features/engineering.py`

In [ ]:
def _as_str(values):
    return values.astype("object").where(values.notna(), MISSING_TOKEN).astype(str)

def frequency_encode(train_values, values):
    freq = _as_str(train_values).value_counts(normalize=True)
    return _as_str(values).map(freq).fillna(0.0).astype("float32")

def label_encode(train_values, values):
    cats = {v: i for i, v in enumerate(sorted(_as_str(train_values).unique()))}
    return _as_str(values).map(cats).fillna(-1).astype("int32")

def split_email_domain(values, prefix):
    parts = _as_str(values).str.split(".")
    return pd.DataFrame(
        {f"{prefix}_provider": parts.str[0], f"{prefix}_suffix": parts.str[-1]},
        index=values.index,
    )

def build_categorical_block(train_df, df, label_cols, freq_cols):
    out = pd.DataFrame(index=df.index)
    for col in label_cols:
        out[f"{col}_le"] = label_encode(train_df[col], df[col])
    for col in freq_cols:
        out[f"{col}_freq"] = frequency_encode(train_df[col], df[col])
    return out

def add_time_features(transaction_dt):
    return pd.DataFrame(
        {
            "tx_hour": ((transaction_dt // 3600) % 24).astype("int32"),
            "tx_dow": ((transaction_dt // 86400) % 7).astype("int32"),
        },
        index=transaction_dt.index,
    )

def add_amount_features(amount):
    cents = (amount - np.floor(amount)).round(2)
    return pd.DataFrame(
        {
            "amt_log1p": np.log1p(amount).astype("float32"),
            "amt_cents": cents.astype("float32"),
        },
        index=amount.index,
    )

def normalize_d_columns(df, transaction_dt, d_cols):
    days = transaction_dt / 86400.0
    out = pd.DataFrame(index=df.index)
    for col in d_cols:
        out[f"{col}_norm"] = (df[col] - days).astype("float32")
    return out

def make_uid(df, card_col="card1", addr_col="addr1", dt_col="TransactionDT", d1_col="D1"):
    day = (df[dt_col] / 86400.0).round()
    ref = (day - df[d1_col]).round().astype("Int64").astype("string").fillna("NA")
    card = df[card_col].astype("Int64").astype("string").fillna("NA")
    addr = df[addr_col].astype("Int64").astype("string").fillna("NA")
    return (card + "_" + addr + "_" + ref).astype("object")

def add_uid_aggregates(df, uid, amount_col="TransactionAmt"):
    work = pd.DataFrame({"uid": uid.to_numpy(), "amt": df[amount_col].to_numpy()})
    grp = work.groupby("uid")["amt"]
    count = grp.transform("count")
    mean = grp.transform("mean")
    std = grp.transform("std").fillna(0.0)
    return pd.DataFrame(
        {
            "uid_count": count.astype("float32").to_numpy(),
            "uid_amt_mean": mean.astype("float32").to_numpy(),
            "uid_amt_std": std.astype("float32").to_numpy(),
            "uid_amt_ratio": (work["amt"] / mean.replace(0.0, np.nan)).astype("float32").to_numpy(),
        },
        index=df.index,
    )

## 1. Load train + test, build UID aggregates over the union

The UID aggregates are label-free, so they are computed once over the train+test union and
then split back. The UID key is also frequency-encoded (fit on train rows only, like the other
categoricals). Everything else matches EXP-003.

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity

test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]
test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

n_train = len(train)
full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
del train, test

# UID + label-free aggregates over the union
uid_full = make_uid(full)
uid_agg_full = add_uid_aggregates(full, uid_full)
full["uid"] = uid_full.to_numpy()
print(f"UID unique: {uid_full.nunique():,} over {len(full):,} rows")

# email splits (row-local)
for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:
    full = pd.concat([full, split_email_domain(full[col], prefix)], axis=1)

In [ ]:
full["DT_M"] = (full["TransactionDT"] / SECONDS_PER_MONTH).astype(int)

numeric_features = [
    c for c in full.columns
    if full[c].dtype != "O" and c not in EXCLUDE_COLS and c not in ("DT_M", "uid")
]
label_cols = [c for c in full.columns if full[c].dtype == "O" and c != "uid"]
freq_cols = label_cols + FREQ_NUMERIC_CATS + ["uid"]  # UID frequency-encoded

row_local = pd.concat(
    [
        add_time_features(full["TransactionDT"]),
        add_amount_features(full["TransactionAmt"]),
        normalize_d_columns(full, full["TransactionDT"], D_NORM_COLS),
        uid_agg_full,
    ],
    axis=1,
)

X_num_full = pd.concat([full[numeric_features].astype("float32"), row_local], axis=1)
cat_source_full = full[label_cols + FREQ_NUMERIC_CATS + ["uid"]].copy()
y = full["isFraud"].to_numpy()  # NaN on test rows
dt = full["TransactionDT"].to_numpy()
months = full["DT_M"].to_numpy()
trans_ids = full["TransactionID"].to_numpy()
del full, row_local, uid_agg_full

# split masks over the union
is_train = np.arange(len(X_num_full)) < n_train
y_tr_all = y[is_train].astype(int)
cutoff = np.quantile(dt[is_train], SPLIT_QUANTILE)
train_mask = is_train & (dt < cutoff)      # Scheme A train
holdout_mask = is_train & (dt >= cutoff)   # Scheme A holdout
print(f"Features: {X_num_full.shape[1]} numeric+agg | train {train_mask.sum():,} | holdout {holdout_mask.sum():,} | test {(~is_train).sum():,}")

def make_X(fit_mask, rows_mask):
    block = build_categorical_block(
        cat_source_full[fit_mask], cat_source_full[rows_mask], label_cols, freq_cols
    )
    base = X_num_full[rows_mask]
    return pd.concat([base.reset_index(drop=True), block.reset_index(drop=True)], axis=1)

## 2. Scheme B - month-wise GroupKFold (7 folds), over training rows only

In [ ]:
train_months_all = months[is_train]
fold_aucs, fold_best_iters = [], []
t0 = time.time()
for m in sorted(set(train_months_all)):
    tr = is_train & (months != m)
    va = is_train & (months == m)
    X_tr = make_X(fit_mask=tr, rows_mask=tr)
    X_va = make_X(fit_mask=tr, rows_mask=va)
    clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
    clf.fit(
        X_tr, y[tr].astype(int),
        eval_set=[(X_va, y[va].astype(int))],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
    )
    auc = roc_auc_score(y[va].astype(int), clf.predict_proba(X_va)[:, 1])
    fold_aucs.append(auc)
    fold_best_iters.append(clf.best_iteration_)
    del X_tr, X_va
    print(f"fold month={m}: AUC={auc:.4f}  best_iter={clf.best_iteration_}  ({(time.time()-t0)/60:.1f} min elapsed)")

scheme_b_mean, scheme_b_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f"\nScheme B GroupKFold: {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")

## 3. Scheme A model - ES inside the train partition, refit at best_iter * 1.1

In [ ]:
es_month = train_months_all.max()
sub_tr = train_mask & (months < es_month)
es_va = train_mask & (months == es_month)

X_sub_tr = make_X(fit_mask=train_mask, rows_mask=sub_tr)
X_es_va = make_X(fit_mask=train_mask, rows_mask=es_va)
es_clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
es_clf.fit(
    X_sub_tr, y[sub_tr].astype(int),
    eval_set=[(X_es_va, y[es_va].astype(int))],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
)
best_iter = es_clf.best_iteration_
final_rounds = max(int(best_iter * 1.1), 100)
del X_sub_tr, X_es_va
print(f"ES month: {es_month} | best_iter: {best_iter} | refit rounds: {final_rounds}")

X_train = make_X(fit_mask=train_mask, rows_mask=train_mask)
model = lgb.LGBMClassifier(n_estimators=final_rounds, **LGB_PARAMS)
model.fit(X_train, y[train_mask].astype(int))
feature_cols = list(X_train.columns)
del X_train

X_holdout = make_X(fit_mask=train_mask, rows_mask=holdout_mask)
val_proba = model.predict_proba(X_holdout)[:, 1]
holdout_auc = roc_auc_score(y[holdout_mask].astype(int), val_proba)
del X_holdout
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (EXP-003: 0.9296)")

## 4. Save holdout predictions and submission

In [ ]:
pd.DataFrame(
    {"TransactionID": trans_ids[holdout_mask], "y_true": y[holdout_mask].astype(int), "score": val_proba}
).to_csv("holdout_pred_exp004.csv", index=False)
print("Saved holdout_pred_exp004.csv")

X_test = make_X(fit_mask=train_mask, rows_mask=~is_train)
assert list(X_test.columns) == feature_cols, "feature contract violated"
test_proba = model.predict_proba(X_test)[:, 1]
pd.DataFrame(
    {"TransactionID": trans_ids[~is_train].astype(int), "isFraud": test_proba}
).to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({(~is_train).sum():,} rows, expected 506,691)")

print("\n=== EXP-004 summary ===")
print(f"Scheme A holdout AUC : {holdout_auc:.4f}")
print(f"Scheme B GroupKFold  : {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")
print(f"Per-fold AUCs        : {[round(float(a), 4) for a in fold_aucs]}")